# Brain Dance - De3DGS Video Processing

Process your own video with **Deformable 3D Gaussians (De3DGS)** on Google Colab.

**Version**: 2.1.0  
**Date**: 2026-02-10  
**Requirements**: Colab (T4/L4 GPU), CUDA 12.8, 12GB+ VRAM  
**Estimated Runtime**: 30-60 minutes depending on video length

## Workflow

1. **Section 0**: Configuration
2. **Section A**: Environment Setup (dependencies, GPU, CUDA kernels)
3. **Section B**: Upload Your Video
4. **Section C**: Process Video with De3DGS
5. **Section D**: Download Results
6. **Section E**: Summary

---
## Section 0: Configuration

In [ ]:
# Cell 0.1: Configuration
"""
All tunable parameters in one place.
Modify these values to customize your run.

Training iterations guide:
- 5,000: Quick validation (~10 min on T4, low quality)
- 20,000: Recommended for real videos (~30 min on T4)
- 40,000: Maximum quality for synthetic/D-NeRF (~60 min on T4)
"""

from dataclasses import dataclass

@dataclass
class NotebookConfig:
    """All tunable parameters in one place."""
    # Repository
    repo_url: str = "https://github.com/ujseah/brain-dance.git"
    branch: str = "main"
    
    # Training
    training_iterations: int = 20000  # Official default for real-world videos (use 5000 for quick validation)
    checkpoint_interval: int = 1000
    
    # VRAM limit
    max_vram_gb: float = 12.0
    
    # Paths
    checkpoint_dir: str = "/content/brain_dance_checkpoint"
    output_dir: str = "/content/brain_dance_output"
    repo_dir: str = "/content/brain-dance"
    
    # COLMAP settings
    colmap_relaxed_mode: bool = True  # Enable for low-parallax videos (tripod/gimbal shots)

CONFIG = NotebookConfig()
print(f"Configuration loaded:")
print(f"  Repository: {CONFIG.repo_url}")
print(f"  Branch: {CONFIG.branch}")
print(f"  Training iterations: {CONFIG.training_iterations}")
print(f"  COLMAP relaxed mode: {CONFIG.colmap_relaxed_mode}")

In [ ]:
# Cell 0.2: Resume Detection
"""
Checkpoint manager for resuming after Colab disconnects.

TROUBLESHOOTING:
If you encounter import errors (e.g., "libonnxruntime.so.1 not found")
after updating the notebook, your cached checkpoint may be stale.
Uncomment the next line to force a fresh installation:
"""
# CHECKPOINT.reset()  # Uncomment to clear all checkpoints and reinstall

import os
import json
from pathlib import Path
from datetime import datetime

class CheckpointManager:
    """Manage notebook execution checkpoints for resume capability."""
    
    def __init__(self, checkpoint_dir: str):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.state_file = self.checkpoint_dir / "state.json"
        self.state = self._load_state()
    
    def _load_state(self) -> dict:
        if self.state_file.exists():
            return json.loads(self.state_file.read_text())
        return {"completed_sections": [], "start_time": None}
    
    def _save_state(self):
        self.state_file.write_text(json.dumps(self.state, indent=2))
    
    def is_complete(self, section: str) -> bool:
        return section in self.state["completed_sections"]
    
    def mark_complete(self, section: str):
        if section not in self.state["completed_sections"]:
            self.state["completed_sections"].append(section)
            self._save_state()
            print(f"Checkpoint saved: {section}")
    
    def reset(self):
        """Clear all checkpoints (for fresh run)."""
        self.state = {"completed_sections": [], "start_time": None}
        self._save_state()
        print("Checkpoints cleared - starting fresh")

CHECKPOINT = CheckpointManager(CONFIG.checkpoint_dir)
print(f"Checkpoint status: {len(CHECKPOINT.state['completed_sections'])} sections complete")
if CHECKPOINT.state['completed_sections']:
    print(f"  Completed: {CHECKPOINT.state['completed_sections']}")
    print("  To start fresh, run: CHECKPOINT.reset()")

---
## Section A: Environment Setup

**Important**: Cell A.1 must run BEFORE any other cells that import torch.

In [ ]:
# Cell A.1: Install Dependencies (MUST run before any torch imports)
"""
Install De3DGS dependencies for Colab CUDA 12.8.

Note: We use Colab's native numpy/Pillow/matplotlib (no downgrade needed).
CUDA extensions compile at runtime against whatever numpy is installed.

pycolmap: Using lyehe/build_gpu_colmap CUDA 12.8 wheels for GPU-accelerated SIFT.
This avoids the CUDA runtime conflict between official pycolmap-cuda12 (CUDA 12.9)
and PyTorch 2.8.0+cu128 (CUDA 12.8).
"""
import subprocess
import sys

# ALWAYS install onnxruntime-gpu (required by lyehe pycolmap CUDA wheels)
# This must be OUTSIDE the checkpoint block because users may have cached
# checkpoints from before this dependency was added.
# v3.14.0-dev1 has ONNX enabled for LightGlue/ALIKED feature matching.
print("Ensuring ONNX Runtime GPU is installed...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"],
               check=False)  # Don't fail if already installed

if CHECKPOINT.is_complete("section_a1_deps"):
    print("[SKIP] Section A.1 already complete, skipping dependency install...")
else:
    print("=" * 50)
    print("INSTALLING De3DGS DEPENDENCIES")
    print("=" * 50)

    # Check if correct PyTorch+CUDA version is already installed
    try:
        result = subprocess.run(
            [sys.executable, "-c",
             "import torch; v=torch.__version__; c=torch.version.cuda or ''; "
             "print(f'{v}|{c}')"],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0:
            version_info = result.stdout.strip()
            torch_ver, cuda_ver = version_info.split('|')
            # Accept 2.8.x with CUDA 12.x
            pytorch_ok = torch_ver.startswith("2.8") and cuda_ver.startswith("12.")
        else:
            pytorch_ok = False
    except Exception:
        pytorch_ok = False

    if pytorch_ok:
        print(f"[OK] PyTorch {torch_ver} with CUDA {cuda_ver} already installed")
    else:
        print("[INFO] Installing PyTorch 2.8.0 with CUDA 12.8 support...")
        print("       (This replaces Colab's default PyTorch)")
        !pip install -q --force-reinstall torch==2.8.0 torchvision==0.23.0 \
            --index-url https://download.pytorch.org/whl/cu128

    # Install other dependencies (using Colab's native numpy/Pillow/matplotlib)
    print("\nInstalling other dependencies...")
    !pip install -q \
        "scipy>=1.12.0" \
        "plyfile>=1.0" \
        "tqdm>=4.66" \
        "lpips>=0.1.4" \
        "imageio>=2.33" \
        "opencv-python" \
        "imageio-ffmpeg" \
        "gdown>=5.1"

    # Install GPU-accelerated pycolmap from lyehe/build_gpu_colmap
    # CUDA 12.8 wheels - compatible with PyTorch 2.8.0+cu128 (no version conflict)
    # See: https://github.com/lyehe/build_gpu_colmap/releases
    print("\nInstalling GPU-accelerated pycolmap (CUDA 12.8)...")
    !pip install -q https://github.com/lyehe/build_gpu_colmap/releases/download/v3.14.0-dev1/pycolmap-3.14.0.dev0+cuda-cp312-cp312-linux_x86_64.whl

    print("\n[OK] Dependencies installed")
    print("\nNote: Using Colab's native numpy/Pillow/matplotlib")
    print("      (CUDA extensions compile against numpy 2.x at runtime)")
    print("      GPU-accelerated COLMAP via lyehe/build_gpu_colmap wheels")
    CHECKPOINT.mark_complete("section_a1_deps")

In [ ]:
# Cell A.2: GPU Verification (now safe to import torch)
import torch

def get_gpu_info() -> dict:
    """Get detailed GPU information."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "NO GPU DETECTED!\n\n"
            "To fix this:\n"
            "1. Go to Runtime > Change runtime type\n"
            "2. Select 'T4 GPU' or 'L4 GPU' under Hardware accelerator\n"
            "3. Click Save and re-run this cell"
        )
    
    props = torch.cuda.get_device_properties(0)
    
    # Detect GPU tier for runtime estimation
    gpu_name = props.name.lower()
    if "a100" in gpu_name:
        tier = "A100"
        estimated_time = "10-15 min"
    elif "l4" in gpu_name:
        tier = "L4"
        estimated_time = "15-25 min"
    elif "v100" in gpu_name:
        tier = "V100"
        estimated_time = "15-20 min"
    elif "t4" in gpu_name:
        tier = "T4"
        estimated_time = "25-35 min"
    else:
        tier = "Unknown"
        estimated_time = "30-45 min"
    
    return {
        "name": props.name,
        "tier": tier,
        "vram_gb": props.total_memory / 1e9,
        "cuda_version": torch.version.cuda,
        "pytorch_version": torch.__version__,
        "estimated_time": estimated_time,
    }

gpu_info = get_gpu_info()
print("=" * 50)
print("GPU VERIFICATION")
print("=" * 50)
print(f"[OK] GPU: {gpu_info['name']}")
print(f"[OK] Tier: {gpu_info['tier']}")
print(f"[OK] VRAM: {gpu_info['vram_gb']:.1f} GB")
print(f"[OK] CUDA: {gpu_info['cuda_version']}")
print(f"[OK] PyTorch: {gpu_info['pytorch_version']}")
print(f"[TIME] Estimated processing time: {gpu_info['estimated_time']}")

if gpu_info['vram_gb'] < CONFIG.max_vram_gb:
    print(f"\n[WARN] VRAM ({gpu_info['vram_gb']:.1f} GB) is below recommended {CONFIG.max_vram_gb} GB")

CHECKPOINT.mark_complete("section_a2_gpu")

In [ ]:
# Cell A.3: Clone Repository
import subprocess
import time

def clone_with_retry(url: str, dest: str, branch: str, max_retries: int = 3):
    """Clone repository with exponential backoff retry."""
    for attempt in range(max_retries):
        try:
            if os.path.exists(dest):
                print(f"Repository already exists at {dest}")
                result = subprocess.run(
                    ["git", "-C", dest, "remote", "get-url", "origin"],
                    capture_output=True, text=True
                )
                if url in result.stdout:
                    print("Updating existing repository...")
                    subprocess.run(["git", "-C", dest, "fetch", "--all"], check=True)
                    subprocess.run(["git", "-C", dest, "checkout", branch], check=True)
                    subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=True)
                    subprocess.run(["git", "-C", dest, "submodule", "update", "--init", "--recursive"], check=True)
                    return True
                else:
                    print("Different repository exists, removing...")
                    subprocess.run(["rm", "-rf", dest], check=True)
            
            print(f"Cloning {url} (attempt {attempt + 1}/{max_retries})...")
            subprocess.run([
                "git", "clone", "--recursive",
                "-b", branch,
                url, dest
            ], check=True)
            return True
        
        except subprocess.CalledProcessError as e:
            wait_time = 2 ** attempt
            print(f"Clone failed, retrying in {wait_time}s...")
            time.sleep(wait_time)
    
    raise RuntimeError(f"Failed to clone repository after {max_retries} attempts")

if CHECKPOINT.is_complete("section_a3_clone"):
    print("[SKIP] Section A.3 already complete, skipping clone...")
else:
    clone_with_retry(
        CONFIG.repo_url,
        CONFIG.repo_dir,
        CONFIG.branch
    )
    os.chdir(CONFIG.repo_dir)
    
    if not os.path.exists("deformable3dgs/train.py"):
        raise RuntimeError("De3DGS submodule not properly initialized!")
    
    print("[OK] Repository cloned and verified")
    CHECKPOINT.mark_complete("section_a3_clone")

In [ ]:
# Cell A.4: Compile De3DGS CUDA Kernels
"""
Compile CUDA kernels for De3DGS. This step:
1. Detects GPU compute capability
2. Applies CUDA 12.x compatibility patches (adds missing headers)
3. Compiles diff-gaussian-rasterization and simple-knn

Evidence-based patches from:
- https://github.com/graphdeco-inria/gaussian-splatting/issues/1215
- https://github.com/graphdeco-inria/gaussian-splatting/issues/1296
- https://github.com/graphdeco-inria/gaussian-splatting/issues/923
"""
import subprocess
import sys
import os

if CHECKPOINT.is_complete("section_a4_cuda"):
    print("[SKIP] Section A.4 already complete")
else:
    print("=" * 50)
    print("COMPILING CUDA KERNELS")
    print("=" * 50)
    print("\nThis may take 5-10 minutes on first run.\n")

    # Detect GPU and CUDA version
    cuda_version = torch.version.cuda
    props = torch.cuda.get_device_properties(0)
    arch = f"{props.major}.{props.minor}"

    print(f"GPU: {props.name}")
    print(f"CUDA: {cuda_version}")
    print(f"Compute capability: {arch}\n")

    # Set environment for compilation
    os.environ["TORCH_CUDA_ARCH_LIST"] = arch
    os.environ["FORCE_CUDA"] = "1"

    setup_script = f"{CONFIG.repo_dir}/scripts/setup_de3dgs.sh"

    if not os.path.exists(setup_script):
        raise FileNotFoundError(f"Setup script not found: {setup_script}")

    # Run with real-time output streaming to see actual errors
    process = subprocess.Popen(
        ["bash", setup_script],
        cwd=CONFIG.repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    # Stream output in real-time
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(
            f"CUDA kernel compilation failed (exit code {process.returncode})\n\n"
            "Check the output above for specific compilation errors."
        )

    print("\n[OK] CUDA kernels compiled successfully")
    CHECKPOINT.mark_complete("section_a4_cuda")

In [ ]:
# Cell A.5: Verify Installation
"""
Verify all CUDA extensions are importable.

Note: Since extensions were installed via subprocess, we need to
refresh Python's import system to see the new packages.
"""
import sys
import importlib

print("=" * 50)
print("INSTALLATION VERIFICATION")
print("=" * 50)

# Refresh import caches to see packages installed by subprocess
importlib.invalidate_caches()

# Add editable install paths to sys.path if not present
submodule_paths = [
    f"{CONFIG.repo_dir}/deformable3dgs/submodules/depth-diff-gaussian-rasterization",
    f"{CONFIG.repo_dir}/deformable3dgs/submodules/simple-knn",
]
for path in submodule_paths:
    if path not in sys.path and os.path.exists(path):
        sys.path.insert(0, path)

errors = 0

try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("[OK] diff-gaussian-rasterization")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")
    errors += 1

try:
    from simple_knn import _C
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")
    errors += 1

try:
    from plyfile import PlyData
    print("[OK] plyfile")
except ImportError as e:
    print(f"[FAIL] plyfile: {e}")
    errors += 1

if errors == 0:
    print("\n[OK] All installations verified!")
    CHECKPOINT.mark_complete("section_a5_verify")
else:
    raise RuntimeError(f"Installation verification failed with {errors} error(s)")

In [ ]:
# Cell A.6: Verify pycolmap Installation
"""
Verify that pycolmap is installed with GPU support.

Using lyehe/build_gpu_colmap CUDA 12.8 wheels for:
- GPU-accelerated SIFT feature extraction (10-50x faster)
- GPU-accelerated feature matching
- Compatible with PyTorch 2.8.0+cu128 (no CUDA runtime conflict)
"""
import importlib
import subprocess
import sys

if CHECKPOINT.is_complete("section_a6_colmap"):
    print("[SKIP] Section A.6 already complete")
else:
    print("=" * 50)
    print("VERIFYING PYCOLMAP INSTALLATION")
    print("=" * 50)

    # Refresh import cache for packages installed by pip in Cell A.1
    importlib.invalidate_caches()

    # Try to import pycolmap
    # Catch both ImportError and RuntimeError because the lyehe wheel raises
    # RuntimeError("Cannot import the C++ backend pycolmap._core") when
    # onnxruntime-gpu is missing (wraps the underlying ImportError)
    try:
        import pycolmap
    except (ImportError, RuntimeError) as e:
        print(f"[WARN] pycolmap import failed: {e}")
        print("[WARN] Installing dependencies...")
        # Install onnxruntime-gpu first (required by lyehe wheels)
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"
        ], check=True)
        # Install pycolmap from lyehe
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "https://github.com/lyehe/build_gpu_colmap/releases/download/v3.14.0-dev1/pycolmap-3.14.0.dev0+cuda-cp312-cp312-linux_x86_64.whl"
        ], check=True)
        importlib.invalidate_caches()
        import pycolmap

    # Check pycolmap version
    print(f"\n[OK] pycolmap version: {pycolmap.__version__}")

    # Check for GPU/CUDA support via Device enum
    has_cuda = hasattr(pycolmap, 'Device') and hasattr(pycolmap.Device, 'cuda')
    
    if has_cuda:
        print("[OK] GPU SIFT: Available (CUDA 12.8 wheels)")
        print("     Feature matching will be 10-50x faster than CPU")
    else:
        print("[WARN] GPU SIFT: Not available")
        print("       Feature matching will use CPU (slower)")

    # Show available functions
    print("\nAvailable pycolmap functions:")
    print("  - pycolmap.extract_features() - SIFT extraction")
    print("  - pycolmap.match_exhaustive() - Feature matching")
    print("  - pycolmap.incremental_mapping() - Sparse reconstruction")
    print("  - pycolmap.undistort_images() - Image undistortion")

    print("\n[OK] pycolmap ready for COLMAP processing")
    CHECKPOINT.mark_complete("section_a6_colmap")

---
## Section B: Upload Your Video

In [ ]:
# Cell B.1: Upload Video
"""
Upload your video to process with De3DGS.

Supported formats: MP4, MOV, AVI, WEBM
Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds

Tips for best results:
- Keep the video short (2-10 seconds)
- Use smooth camera motion
- Avoid fast motion blur
- Good lighting helps quality
"""
from google.colab import files
import subprocess

USER_VIDEO_DIR = f"{CONFIG.output_dir}/user_video"
USER_VIDEO_PATH = None

print("=" * 50)
print("VIDEO UPLOAD")
print("=" * 50)
print("\nSupported formats: MP4, MOV, AVI, WEBM")
print("Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds\n")

uploaded = files.upload()

if uploaded:
    os.makedirs(USER_VIDEO_DIR, exist_ok=True)
    
    for filename, data in uploaded.items():
        video_path = f"{USER_VIDEO_DIR}/{filename}"
        with open(video_path, 'wb') as f:
            f.write(data)
        
        # Get video info
        result = subprocess.run([
            'ffprobe', '-v', 'quiet', '-print_format', 'json',
            '-show_format', '-show_streams', video_path
        ], capture_output=True, text=True)
        
        if result.returncode == 0:
            import json as json_lib
            info = json_lib.loads(result.stdout)
            duration = float(info['format'].get('duration', 0))
            video_stream = next((s for s in info['streams'] if s['codec_type'] == 'video'), {})
            width = video_stream.get('width', 'unknown')
            height = video_stream.get('height', 'unknown')
            fps = eval(video_stream.get('r_frame_rate', '30/1'))
            
            print(f"\n[OK] Video uploaded: {filename}")
            print(f"     Resolution: {width}x{height}")
            print(f"     Duration: {duration:.1f} seconds")
            print(f"     Frame rate: {fps:.1f} fps")
            print(f"     Estimated frames: {int(duration * fps)}")
            
            USER_VIDEO_PATH = video_path
        else:
            print(f"[WARN] Could not read video metadata for {filename}")
            USER_VIDEO_PATH = video_path
        
        # Display first frame preview
        from IPython.display import display, Image as IPImage
        preview_path = f"{USER_VIDEO_DIR}/preview.jpg"
        subprocess.run([
            'ffmpeg', '-y', '-i', video_path, '-vframes', '1',
            '-q:v', '2', preview_path
        ], capture_output=True)
        
        if os.path.exists(preview_path):
            print("\nFirst frame preview:")
            display(IPImage(filename=preview_path, width=400))
else:
    print("[WARN] No video uploaded. Please upload a video to continue.")
    USER_VIDEO_PATH = None

---
## Section C: Process Video with De3DGS

In [ ]:
# Cell C.1: Extract Frames and Prepare Dataset
"""
Extract frames from your video and prepare for De3DGS training.
This step converts your video into the format De3DGS expects.

De3DGS expects images in: source_path/input/*.png
Frame naming MUST be numeric only (0001.png, not frame_0001.png)
because the training code uses int(image_name) for temporal indexing.
"""
import subprocess
from pathlib import Path

if USER_VIDEO_PATH is None:
    raise RuntimeError("No video uploaded! Please run Section B first.")

FRAMES_DIR = f"{CONFIG.output_dir}/frames"
INPUT_DIR = f"{FRAMES_DIR}/input"  # De3DGS expects images in input/ subdirectory
os.makedirs(INPUT_DIR, exist_ok=True)

print("=" * 50)
print("EXTRACTING FRAMES")
print("=" * 50)

# Extract frames with NUMERIC-ONLY naming (required by De3DGS)
# The training code does: fid = int(image_name) / (num_frames - 1)
# So filenames must be pure integers like 0001.png, not frame_0001.png
result = subprocess.run([
    'ffmpeg', '-y', '-i', USER_VIDEO_PATH,
    '-qscale:v', '2',
    f'{INPUT_DIR}/%04d.png'
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"[FAIL] Frame extraction failed: {result.stderr}")
    raise RuntimeError("Frame extraction failed")

frames = sorted(Path(INPUT_DIR).glob("*.png"))
print(f"[OK] Extracted {len(frames)} frames to {INPUT_DIR}")

# Display sample frames
import matplotlib.pyplot as plt
from PIL import Image

if len(frames) >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    indices = [0, len(frames)//2, -1]
    for ax, idx in zip(axes, indices):
        img = Image.open(frames[idx])
        ax.imshow(img)
        ax.set_title(f"Frame {idx if idx >= 0 else len(frames)+idx}")
        ax.axis('off')
    plt.suptitle("Sample Frames from Your Video")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell C.2: Run COLMAP for Camera Poses (via pycolmap)
"""
Extract camera poses from your video frames using GPU-accelerated SIFT.

Uses lyehe/build_gpu_colmap CUDA 12.8 wheels for 10-50x faster matching.

RELAXED MODE (CONFIG.colmap_relaxed_mode):
  - Enabled by default for low-parallax videos (tripod/gimbal shots)
  - Lowers triangulation angle threshold from 16° to 4°
  - Reduces minimum inlier requirement from 100 to 15
  - WARNING: May produce lower quality 3D reconstruction
"""
import importlib
import os
import shutil
from pathlib import Path

# Refresh import cache and import pycolmap
importlib.invalidate_caches()
import pycolmap

print("=" * 50)
print("RUNNING COLMAP (via pycolmap)")
print("=" * 50)
print("\nThis step extracts camera poses from your video.\n")

# Check for CUDA support via Device enum
try:
    has_cuda = hasattr(pycolmap, 'Device') and hasattr(pycolmap.Device, 'cuda')
    device = pycolmap.Device.cuda if has_cuda else pycolmap.Device.cpu
    if has_cuda:
        print("[INFO] Using GPU-accelerated SIFT (CUDA 12.8)")
    else:
        print("[INFO] Using CPU SIFT (GPU not available)")
except:
    has_cuda = False
    device = None
    print("[INFO] Using CPU SIFT")

sparse_dir = f"{FRAMES_DIR}/sparse/0"
database_path = f"{FRAMES_DIR}/distorted/database.db"
image_path = f"{FRAMES_DIR}/input"

# Check cache - skip if already processed
if os.path.exists(sparse_dir) and os.listdir(sparse_dir):
    print(f"[SKIP] COLMAP output already exists at {sparse_dir}")
    print("       Delete this directory to force re-run")
else:
    # Create directories
    os.makedirs(f"{FRAMES_DIR}/distorted/sparse", exist_ok=True)
    os.makedirs(os.path.dirname(database_path), exist_ok=True)

    # Step 1: Feature extraction
    print("[1/4] Extracting features...")
    
    extract_kwargs = {
        "database_path": database_path,
        "image_path": image_path,
        "camera_mode": pycolmap.CameraMode.SINGLE,
    }
    if device is not None:
        extract_kwargs["device"] = device
    
    pycolmap.extract_features(**extract_kwargs)
    print("       Done")

    # Step 2: Feature matching
    print("[2/4] Matching features...")
    
    match_kwargs = {"database_path": database_path}
    if device is not None:
        match_kwargs["device"] = device
    
    pycolmap.match_exhaustive(**match_kwargs)
    print("       Done")

    # Step 3: Sparse reconstruction
    print("[3/4] Running sparse reconstruction...")
    
    # pycolmap 3.14.0: Use IncrementalPipelineOptions
    options = pycolmap.IncrementalPipelineOptions()
    
    if CONFIG.colmap_relaxed_mode:
        print("       [RELAXED MODE] Using lower thresholds for low-parallax videos")
        # Mapper-specific options (nested under .mapper)
        options.mapper.init_min_tri_angle = 4.0
        options.mapper.init_min_num_inliers = 15
        # BA refinement options are direct attributes of IncrementalPipelineOptions
        options.ba_refine_focal_length = False
        options.ba_refine_extra_params = False
    
    # Run incremental mapping
    output_sparse = f"{FRAMES_DIR}/distorted/sparse"
    maps = pycolmap.incremental_mapping(
        database_path=database_path,
        image_path=image_path,
        output_path=output_sparse,
        options=options
    )
    
    if not maps:
        print("\n[FAIL] Sparse reconstruction failed")
        print("\nPossible causes:")
        print("  - Video has insufficient camera translation (parallax)")
        print("  - Too few distinct features in the scene")
        print("  - Motion blur from fast camera movement")
        print("\nTips:")
        print("  - Use a video where the camera physically moves through space")
        print("  - Ensure CONFIG.colmap_relaxed_mode = True (currently: {})".format(CONFIG.colmap_relaxed_mode))
        raise RuntimeError("Sparse reconstruction failed")
    
    print(f"       Done - reconstructed {len(maps)} model(s)")

    # Check if sparse reconstruction was created
    distorted_sparse = f"{FRAMES_DIR}/distorted/sparse/0"
    if not os.path.exists(distorted_sparse):
        print("\n[FAIL] COLMAP mapper did not produce output")
        raise RuntimeError("No sparse reconstruction produced")

    # Step 4: Undistort images
    print("[4/4] Undistorting images...")
    
    pycolmap.undistort_images(
        output_path=FRAMES_DIR,
        input_path=distorted_sparse,
        image_path=image_path,
        output_type="COLMAP"
    )
    print("       Done")

    # Fix directory structure: De3DGS expects sparse/0/, but undistort creates sparse/
    # See: deformable3dgs/scene/dataset_readers.py:174-202
    sparse_base = f"{FRAMES_DIR}/sparse"
    sparse_model = f"{FRAMES_DIR}/sparse/0"
    
    if os.path.exists(sparse_base) and not os.path.exists(sparse_model):
        # Files are directly in sparse/, need to move to sparse/0/
        print("       [FIX] Moving sparse/ to sparse/0/ for De3DGS compatibility...")
        temp_dir = f"{FRAMES_DIR}/sparse_temp"
        shutil.move(sparse_base, temp_dir)
        os.makedirs(sparse_base, exist_ok=True)
        shutil.move(temp_dir, sparse_model)
        print("       Done")

# Verify output
if not os.path.exists(sparse_dir) or not os.listdir(sparse_dir):
    print(f"\n[FAIL] COLMAP did not produce sparse reconstruction")
    print(f"       Expected: {sparse_dir}")
    print("\nPossible causes:")
    print("  - Video has insufficient camera translation (parallax)")
    print("  - Too few distinct features in the scene")
    print("  - Motion blur from fast camera movement")
    print("\nTry:")
    print("  - Use a video where the camera physically moves through space")
    print("  - Set CONFIG.colmap_relaxed_mode = True (if not already)")
    raise RuntimeError("COLMAP sparse reconstruction not found")

sparse_files = os.listdir(sparse_dir)
print(f"\n[OK] COLMAP processing complete")
print(f"     Sparse reconstruction: {sparse_dir}")
print(f"     Files: {sparse_files}")

In [ ]:
# Cell C.3: Train De3DGS on Your Video
"""
Train Deformable 3D Gaussians on your video.
This creates a 4D representation of your scene.

Official recommendations:
- Real-world videos: 20,000 iterations
- Synthetic (D-NeRF): 40,000 iterations
- Quick validation: 5,000 iterations (lower quality)
"""
import time
import subprocess
import sys

DE3DGS_DIR = f"{CONFIG.repo_dir}/deformable3dgs"
TRAINING_OUTPUT = f"{CONFIG.output_dir}/training"

print("=" * 50)
print("TRAINING De3DGS")
print("=" * 50)
print(f"\nIterations: {CONFIG.training_iterations}")
print(f"Output: {TRAINING_OUTPUT}")
print(f"Estimated time: {gpu_info['estimated_time']}")

# Quality warning for low iteration counts
if CONFIG.training_iterations < 20000:
    print(f"\n[WARN] Low iteration count ({CONFIG.training_iterations})")
    print("       For production quality, use 20000+ iterations")
    print("       Current setting is suitable for validation only")
print()

# Verify COLMAP output exists
sparse_dir = f"{FRAMES_DIR}/sparse/0"
if not os.path.exists(sparse_dir):
    raise RuntimeError(
        f"COLMAP sparse reconstruction not found at {sparse_dir}\n"
        "Run Cell C.2 first to generate camera poses"
    )

start_time = time.time()

# Build training command with save_iterations to ensure checkpoint is saved
train_cmd = [
    "python", "train.py",
    "-s", FRAMES_DIR,
    "-m", TRAINING_OUTPUT,
    "--iterations", str(CONFIG.training_iterations),
    "--save_iterations", str(CONFIG.training_iterations),  # Save at final iteration
]

print(f"Command: {' '.join(train_cmd)}")
print("-" * 50)

# Run training with real-time output
process = subprocess.Popen(
    train_cmd,
    cwd=DE3DGS_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

process.wait()

print("-" * 50)

elapsed = time.time() - start_time

if process.returncode != 0:
    print(f"\n[FAIL] Training failed (exit code {process.returncode})")
    raise RuntimeError("De3DGS training failed - check output above")

# Verify checkpoint was saved
point_cloud_dir = f"{TRAINING_OUTPUT}/point_cloud"
if not os.path.exists(point_cloud_dir):
    print(f"\n[WARN] No checkpoint saved at {point_cloud_dir}")
    print("       Training may have failed silently")
else:
    checkpoints = os.listdir(point_cloud_dir)
    print(f"\n[OK] Training completed in {elapsed/60:.1f} minutes")
    print(f"     Checkpoints saved: {checkpoints}")

In [ ]:
# Cell C.4: Render Results
"""
Render the trained De3DGS model into an MP4 video.
"""
from pathlib import Path
import subprocess
import sys

RENDER_OUTPUT = f"{CONFIG.output_dir}/renders"
os.makedirs(RENDER_OUTPUT, exist_ok=True)

print("=" * 50)
print("RENDERING RESULTS")
print("=" * 50)

training_output = Path(TRAINING_OUTPUT)
point_cloud_dir = training_output / "point_cloud"

# Check if checkpoint exists before rendering
if not point_cloud_dir.exists():
    print(f"\n[FAIL] No model checkpoint found!")
    print(f"       Expected: {point_cloud_dir}")
    print("\n       Run Cell C.3 (training) first to create a checkpoint.")
    raise RuntimeError("No checkpoint found - cannot render")

# Show available checkpoints
checkpoints = list(point_cloud_dir.iterdir())
print(f"\nFound checkpoints: {[c.name for c in checkpoints]}")

# Run De3DGS renderer
render_script = f"{CONFIG.repo_dir}/deformable3dgs/render.py"

if not os.path.exists(render_script):
    raise FileNotFoundError(f"render.py not found at {render_script}")

print("\nRendering frames...")
print("-" * 50)

# Run with real-time output
process = subprocess.Popen(
    ["python", render_script, "-m", str(training_output)],
    cwd=f"{CONFIG.repo_dir}/deformable3dgs",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

process.wait()

print("-" * 50)

if process.returncode != 0:
    print(f"\n[FAIL] Rendering failed (exit code {process.returncode})")
    raise RuntimeError("Rendering failed - check output above")

# Find rendered frames
render_dir = training_output / "train" / f"ours_{CONFIG.training_iterations}" / "renders"
if not render_dir.exists():
    render_dir = training_output / "test" / f"ours_{CONFIG.training_iterations}"

if render_dir.exists():
    render_frames = sorted(render_dir.glob("*.png"))

    if render_frames:
        print(f"\n[OK] Rendered {len(render_frames)} frames")

        # Create MP4
        mp4_path = f"{RENDER_OUTPUT}/de3dgs_render.mp4"
        !ffmpeg -y -framerate 15 \
            -pattern_type glob -i '{render_dir}/*.png' \
            -c:v libx264 -pix_fmt yuv420p \
            -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
            {mp4_path} 2>/dev/null

        if os.path.exists(mp4_path):
            mp4_size = os.path.getsize(mp4_path) / 1e6
            print(f"[OK] Rendered video: {mp4_path} ({mp4_size:.1f} MB)")

            from IPython.display import Video, display
            print("\nPlaying rendered video:")
            display(Video(mp4_path, embed=True, width=640))
        else:
            print("[WARN] Failed to create MP4 from rendered frames")
    else:
        print("[WARN] No rendered frames found in output directory")
else:
    print(f"[WARN] Render directory not found: {render_dir}")

---
## Section D: Download Results

In [ ]:
# Cell D.1: Package Results into ZIP
"""
Package PLY files, MP4 videos, and model checkpoints
into a single ZIP file for download.
"""
import zipfile
from pathlib import Path

DOWNLOAD_ZIP = "/content/brain_dance_results.zip"

print("=" * 50)
print("PACKAGING RESULTS")
print("=" * 50)

files_added = 0

with zipfile.ZipFile(DOWNLOAD_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    
    # Add PLY files
    print("\nAdding PLY files...")
    for ply_file in Path(CONFIG.output_dir).rglob("*.ply"):
        arcname = f"ply_files/{ply_file.parent.name}/{ply_file.name}"
        zf.write(ply_file, arcname)
        files_added += 1
        if files_added <= 5:
            print(f"  + {arcname}")
    if files_added > 5:
        print(f"  ... and {files_added - 5} more PLY files")
    
    # Add MP4 renders
    print("\nAdding MP4 videos...")
    mp4_count = 0
    for mp4_file in Path(CONFIG.output_dir).rglob("*.mp4"):
        arcname = f"videos/{mp4_file.name}"
        zf.write(mp4_file, arcname)
        files_added += 1
        mp4_count += 1
        print(f"  + {arcname} ({os.path.getsize(mp4_file) / 1e6:.1f} MB)")
    
    if mp4_count == 0:
        print("  (No MP4 files found)")
    
    # Add model checkpoints
    print("\nAdding model checkpoints...")
    pth_count = 0
    for pth_file in Path(CONFIG.output_dir).rglob("*.pth"):
        arcname = f"models/{pth_file.parent.name}/{pth_file.name}"
        zf.write(pth_file, arcname)
        files_added += 1
        pth_count += 1
        print(f"  + {arcname} ({os.path.getsize(pth_file) / 1e6:.1f} MB)")
    
    if pth_count == 0:
        print("  (No .pth files found)")

zip_size_mb = os.path.getsize(DOWNLOAD_ZIP) / 1e6
print(f"\n" + "=" * 50)
print(f"[OK] Created: {DOWNLOAD_ZIP}")
print(f"     Total files: {files_added}")
print(f"     Size: {zip_size_mb:.1f} MB")
print("=" * 50)

In [ ]:
# Cell D.2: Download Results
"""
Download the packaged ZIP file to your local machine.
Your browser will prompt you to save the file.
"""
from google.colab import files

print("=" * 50)
print("DOWNLOAD RESULTS")
print("=" * 50)
print()

if os.path.exists(DOWNLOAD_ZIP):
    zip_size = os.path.getsize(DOWNLOAD_ZIP) / 1e6
    print(f"Downloading: brain_dance_results.zip")
    print(f"Size: {zip_size:.1f} MB")
    print()
    print("Your browser will prompt you to save the file...")
    print()
    
    files.download(DOWNLOAD_ZIP)
    
    print("\n[OK] Download initiated!")
    print()
    print("ZIP contents:")
    print("  ply_files/  - PLY Gaussian files")
    print("  videos/     - Rendered MP4 videos")
    print("  models/     - Model checkpoints (.pth)")
else:
    print("[FAIL] No results ZIP found.")
    print("       Run Cell D.1 first to package the results.")

---
## Section E: Summary

In [ ]:
# Cell E.1: Final Summary
"""
Display final processing summary.
"""
import torch

print("=" * 60)
print("DE3DGS PROCESSING SUMMARY")
print("=" * 60)
print()

# Check completed sections
env_ok = CHECKPOINT.is_complete("section_a5_verify")
print(f"Environment Setup:  {'PASS' if env_ok else 'FAIL'}")

video_ok = USER_VIDEO_PATH is not None
print(f"Video Uploaded:     {'PASS' if video_ok else 'SKIP'}")

training_ok = os.path.exists(f"{CONFIG.output_dir}/training")
print(f"Training Complete:  {'PASS' if training_ok else 'SKIP'}")

results_ok = os.path.exists(DOWNLOAD_ZIP)
print(f"Results Packaged:   {'PASS' if results_ok else 'SKIP'}")

print()

# Memory usage
if torch.cuda.is_available():
    max_allocated = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Peak GPU Memory: {max_allocated:.1f} GB / {total:.1f} GB ({max_allocated/total*100:.0f}%)")

print()
print("=" * 60)
if env_ok and video_ok and training_ok and results_ok:
    print("PROCESSING COMPLETE - Download your results from Section D")
else:
    print("PROCESSING INCOMPLETE - Check the sections above")
print("=" * 60)